# Behavior Encoding GLMs For Single Neurons (Parallel)

This notebook performs the same negative-binomial GLM screens as `choice_encoding_glm.ipynb`, but the expensive track, event, and position screens are distributed across CPU worker processes. The statistical model, feature groups, CV splits, FDR correction, and downstream plots are kept aligned with the original notebook.

The GPUs are not used by `statsmodels` GLM fitting. On this server the practical acceleration comes from running many independent small GLMs in parallel across the available CPU cores while keeping BLAS threads capped inside each worker.


## Imports

Set up the project paths, plotting defaults, and statistical libraries used by the GLM workflow.


In [ ]:
import os

# Must be set before importing numpy/scipy/statsmodels in a fresh kernel.
# Each worker runs one BLAS thread; parallelism happens at the process level.
for _var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[_var] = "1"

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.tools.sm_exceptions import PerfectSeparationError
import plotly.graph_objects as go
from plotly.subplots import make_subplots

project_root = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "analysisVR").is_dir() and (p / "baseVR").is_dir()
)
analysis_root = project_root / "analysisVR"
notebook_dir = analysis_root / "scripted_plotting" / "animal_6_analysis"
for _path in (project_root, analysis_root, notebook_dir):
    if str(_path) not in sys.path:
        sys.path.append(str(_path))

from baseVR.base_functionality import init_import_paths
init_import_paths()

from analytics_processing import analytics
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames, sessionlist_fullfnames_from_args
from CustomLogger import CustomLogger as Logger
from choice_encoding_glm_parallel_utils import (
    add_panel_fdr as add_panel_fdr_parallel,
    configure_single_thread_blas,
    run_screen_parallel,
)

configure_single_thread_blas()
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 80)
Logger().init_logger(None, None, logging_level="WARNING")


## Configuration

Choose the animal/session subset, event windows, model thresholds, and quick-test limits. The default is the full analysis; set `QUICK_TEST = True` for a fast smoke test.


In [ ]:

ANIMAL_IDS = [6]
PARADIGM_IDS = [1100]
SESSION_IDS = None
EXCL_SESSION_NAMES = [
    "2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min",
    "2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min",
    "2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min",
]

BIN_SECONDS = 0.04
BIN_US = 40_000
POSITION_BIN_SIZE_CM = 10

EVENT_SPECS = {
    "cueZone_visible": {"label": "Cue visible", "left": -3, "right": 37},
    "cueZone_exit": {"label": "Cue exit", "left": -10, "right": 30},
    "enter_reward1Zone": {"label": "R1 entry", "left": -10, "right": 30},
    "enter_reward2Zone": {"label": "R2 entry", "left": -10, "right": 30},
    "exit_reward1Zone": {"label": "R1 exit", "left": -10, "right": 30},
    "exit_reward2Zone": {"label": "R2 exit", "left": -10, "right": 30},
    "reward1_sound": {"label": "Reward 1 sound", "left": -1, "right": 39},
    "reward2_sound": {"label": "Reward 2 sound", "left": -1, "right": 39},
}
EVENT_ORDER = list(EVENT_SPECS)
EVENT_LABELS = {k: v["label"] for k, v in EVENT_SPECS.items()}

FEATURE_GROUPS = {
    "task": ["cue_binary", "choice_side_binary", "outcome_binary"],
    "choice_state": ["upcoming_choice_skip", "upcoming_choice_stop"],
    "reward_state": ["cue_visible_1", "cue_visible_2", "reward_window_pre", "reward_window_post", "reward_sound", "reward_valve"],
    "movement": ["speed", "acceleration", "rotation", "rotation_acceleration", "total_motion", "total_acceleration", "forward_prop", "forward_rotation_corr"],
    "pose": ["head_angle", "head_angle_velocity", "movement_energy", "pose_angle", "pose_angle_velocity", "body_angle", "body_angle_velocity"],
    "licking": ["lick"],
}
FEATURE_ORDER = list(FEATURE_GROUPS)
FEATURE_COLORS = {
    "task": "#386641",
    "choice_state": "#bc4749",
    "reward_state": "#f2a65a",
    "movement": "#277da1",
    "pose": "#6a4c93",
    "licking": "#595959",
    "position": "#111111",
}

CONTINUOUS_COLS = {
    "speed", "acceleration", "rotation", "rotation_acceleration", "total_motion", "total_acceleration",
    "forward_prop", "forward_rotation_corr", "head_angle", "head_angle_velocity", "movement_energy",
    "pose_angle", "pose_angle_velocity", "body_angle", "body_angle_velocity", "trial_number_z",
}

MIN_TRIALS = 18
MIN_ROWS = 18
MIN_MEAN_COUNT = 0.02
MIN_NULL_DEVIANCE = 0.1
MAX_ABS_FDE = 10.0
N_CV_FOLDS = 5
FDR_ALPHA = 0.05
RANDOM_STATE = 42
PROGRESS_EVERY = 500

QUICK_TEST = False
MAX_TRACK_FITS = None
MAX_EVENT_FITS = None
MAX_POSITION_FITS = None
if QUICK_TEST:
    MAX_TRACK_FITS = 100
    MAX_EVENT_FITS = 100
    MAX_POSITION_FITS = 40

# Parallel execution settings. statsmodels is CPU-only, so this notebook uses
# process-level parallelism. Stop competing notebooks before pushing this high.
CPU_COUNT = os.cpu_count() or 1
N_WORKERS = min(176, max(1, CPU_COUNT - 16))
GROUPS_PER_TASK = 1
PARALLEL_START_METHOD = "fork"
USE_RESULT_CACHE = True
FORCE_REFIT = False
RESULT_CACHE_DIR = notebook_dir / "choice_encoding_glm_parallel_cache"



## Load Analytics

Load firing-rate, behavior, event, and optional unit metadata analytics. `Behavior40msAligned` is the source of truth for cue, choice, and outcome predictors.


In [ ]:

def _ensure_flat(df):
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out


def _unit_cols(df):
    return sorted([c for c in df.columns if str(c).startswith("Unit")])


def _normalize_unit(series):
    text = series.astype(str).str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    nums = pd.to_numeric(text, errors="coerce").combine_first(pd.to_numeric(series, errors="coerce"))
    return nums.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


def _build_meta_map(spike_df):
    if spike_df is None:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])
    meta = _ensure_flat(spike_df)
    area_col = next((c for c in ["fine_brain_area", "brain_area", "region", "area"] if c in meta.columns), None)
    unit_col = next((c for c in ["unit", "unit_id", "unit_name", "cluster_id_str", "cluster_id", "entry_id"] if c in meta.columns), None)
    if unit_col is None:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])
    meta["unit"] = _normalize_unit(meta[unit_col])
    meta["brain_region"] = meta[area_col].astype(str) if area_col else "Unknown"
    meta["brain_region"] = meta["brain_region"].replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"}).fillna("Unknown")
    return meta.dropna(subset=["session_id", "unit"]).groupby(["session_id", "unit"], as_index=False)["brain_region"].first()


session_dirs, _ = sessionlist_fullfnames_from_args(
    PARADIGM_IDS, ANIMAL_IDS, SESSION_IDS, excl_session_names=EXCL_SESSION_NAMES
)
session_names = fullfnames2snames(session_dirs)

behavior_40_raw = analytics.get_analytics("Behavior40msAligned", session_names=session_names)
behavior_track_raw = analytics.get_analytics("BehaviorTrackwise", session_names=session_names)
fr40_raw = analytics.get_analytics("FiringRate40msHz", session_names=session_names)
fr_track_raw = analytics.get_analytics("FiringRateTrackwiseHz", session_names=session_names)
t0_events_raw = analytics.get_analytics("TrialWiseT0Events40ms", session_names=session_names)
spike_meta_raw = analytics.get_analytics("SpikeClusterMetadata", session_names=session_names)

behavior_40 = _ensure_flat(behavior_40_raw)
behavior_track = _ensure_flat(behavior_track_raw)
fr40 = _ensure_flat(fr40_raw)
fr_track = _ensure_flat(fr_track_raw)
t0_events = _ensure_flat(t0_events_raw)
meta_map = _build_meta_map(spike_meta_raw)
region_lookup = meta_map.set_index(["session_id", "unit"])["brain_region"].to_dict()

unit_cols = sorted(set(_unit_cols(fr40)) & set(_unit_cols(fr_track)))
print(f"Sessions requested: {len(session_names)}")
print(f"Sessions with 40 ms behavior: {behavior_40['session_id'].nunique()}")
print(f"Sessions with trackwise FR: {fr_track['session_id'].nunique()}")
print(f"Units in both FR tables: {len(unit_cols)}")


## Predictor Engineering

Create canonical task, state, movement, pose, and licking predictors. Task variables are derived from `Behavior40msAligned`, then merged into trackwise and event-aligned tables.


In [ ]:

def _first_valid(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def _zscore_by_session(df, value_col):
    def zscore(s):
        std = s.std(ddof=0)
        return (s - s.mean()) / std if pd.notna(std) and std > 0 else pd.Series(0.0, index=s.index)
    return df.groupby("session_id", group_keys=False)[value_col].apply(zscore)


def add_task_features(df):
    cue = pd.to_numeric(df.get("cue"), errors="coerce")
    r1 = pd.to_numeric(df.get("choice_R1"), errors="coerce").fillna(0).gt(0)
    r2 = pd.to_numeric(df.get("choice_R2"), errors="coerce").fillna(0).gt(0)
    outcome = pd.to_numeric(df.get("trial_outcome"), errors="coerce")
    df["cue_binary"] = np.where(cue.eq(1), 0.0, np.where(cue.eq(2), 1.0, np.nan))
    df["choice_side_binary"] = np.where(r1 & ~r2, 0.0, np.where(r2 & ~r1, 1.0, np.nan))
    df["outcome_binary"] = np.where(outcome.notna(), outcome.gt(0).astype(float), np.nan)
    return df


def add_state_features(df):
    if "upcoming_choice" in df.columns:
        upcoming = pd.to_numeric(df["upcoming_choice"], errors="coerce")
        df["upcoming_choice_skip"] = upcoming.lt(0).astype(float)
        df["upcoming_choice_stop"] = upcoming.gt(0).astype(float)
    if "reward_window" in df.columns:
        reward_window = pd.to_numeric(df["reward_window"], errors="coerce")
        df["reward_window_pre"] = reward_window.lt(0).astype(float)
        df["reward_window_post"] = reward_window.gt(0).astype(float)
    if "cue_visible" in df.columns:
        cue_visible = pd.to_numeric(df["cue_visible"], errors="coerce")
        df["cue_visible_1"] = cue_visible.eq(1).astype(float)
        df["cue_visible_2"] = cue_visible.eq(2).astype(float)
    for raw_col, out_col in [("lick_detected", "lick"), ("reward-sound_detected", "reward_sound"), ("reward-valve-open_detected", "reward_valve")]:
        if raw_col in df.columns:
            df[out_col] = pd.to_numeric(df[raw_col], errors="coerce").fillna(0).clip(0, 1)
    return df


EVENT_RENAMES = {
    "frame_raw_500msMedian": "speed",
    "frame_raw_abs_acc_500msMedian": "acceleration",
    "frame_YawPitch_abs_vel_sum_500msMedian": "rotation",
    "frame_YawPitch_abs_acc_sum_500msMedian": "rotation_acceleration",
    "frame_RawYawPitch_abs_vel_sum_500msMedian": "total_motion",
    "frame_RawYawPitch_abs_acc_sum_500msMedian": "total_acceleration",
    "frame_forward_prop": "forward_prop",
    "forward_vs_rotation_corr": "forward_rotation_corr",
    "head_angle_vel": "head_angle_velocity",
    "movement_energy_smooth5": "movement_energy",
}
TRACK_RENAMES = {
    "posbin_raw_500msMedian": "speed",
    "posbin_raw_abs_acc_500msMedian": "acceleration",
    "posbin_YawPitch_abs_vel_sum_500msMedian": "rotation",
    "posbin_YawPitch_abs_acc_sum_500msMedian": "rotation_acceleration",
    "posbin_RawYawPitch_abs_vel_sum_500msMedian": "total_motion",
    "posbin_RawYawPitch_abs_acc_sum_500msMedian": "total_acceleration",
    "posbin_forward_prop": "forward_prop",
    "forward_vs_rotation_corr": "forward_rotation_corr",
    "facecam_pose_nose_neck_body1_angle": "pose_angle",
    "facecam_pose_nose_neck_body1_angle_velocity": "pose_angle_velocity",
    "facecam_pose_body1_body2_body3_angle": "body_angle",
    "facecam_pose_body1_body2_body3_angle_velocity": "body_angle_velocity",
}


def prepare_behavior_40(df):
    out = df.rename(columns=EVENT_RENAMES).copy()
    out["from_ephys_timestamp"] = pd.to_numeric(out["from_ephys_timestamp"], errors="coerce")
    out["to_ephys_timestamp"] = pd.to_numeric(out["to_ephys_timestamp"], errors="coerce")
    out["trial_id"] = pd.to_numeric(out["trial_id"], errors="coerce")
    out = add_state_features(add_task_features(out))
    return out


def prepare_track_behavior(df):
    out = df.rename(columns=TRACK_RENAMES).copy()
    out["trial_id"] = pd.to_numeric(out["trial_id"], errors="coerce")
    out["from_position_bin"] = pd.to_numeric(out["from_position_bin"], errors="coerce")
    if "cue_visible" not in out.columns and {"track_zone", "cue"}.issubset(out.columns):
        cue = pd.to_numeric(out["cue"], errors="coerce")
        cue_zone = out["track_zone"].astype(str).isin(["visibleCue", "nextToCue"])
        out["cue_visible"] = np.where(cue_zone, cue, 0.0)
    out = add_state_features(add_task_features(out))
    return out


behavior_40_features = prepare_behavior_40(behavior_40)
behavior_track_features = prepare_track_behavior(behavior_track)

task_cols = ["cue_binary", "choice_side_binary", "outcome_binary"]
trial_task_table = (
    behavior_40_features.loc[behavior_40_features["trial_id"].ge(0), ["session_id", "trial_id", *task_cols]]
    .groupby(["session_id", "trial_id"], as_index=False)
    .agg(_first_valid)
    .sort_values(["session_id", "trial_id"])
)
trial_task_table["trial_number"] = trial_task_table.groupby("session_id").cumcount() + 1
trial_task_table["trial_number_z"] = _zscore_by_session(trial_task_table, "trial_number")

display(trial_task_table.head())


## Build Model Tables

Build compact wide model tables with one count column per unit. During fitting, each wide table is converted to a single-unit tidy frame, avoiding a very large all-unit long table in memory.


In [ ]:

def _safe_counts(values):
    return np.rint(np.clip(pd.to_numeric(values, errors="coerce").fillna(0).to_numpy(dtype=float), 0, None)).astype(np.int32)


def _available_feature_groups(df, groups):
    return {name: [c for c in cols if c in df.columns] for name, cols in groups.items() if any(c in df.columns for c in cols)}


def _expand_event_index(events):
    rows = []
    keep = events.dropna(subset=["session_id", "trial_id", "t0_event_name", "t0"]).copy()
    keep = keep[keep["t0_event_name"].isin(EVENT_SPECS)]
    for row in keep.itertuples(index=False):
        spec = EVENT_SPECS[row.t0_event_name]
        t0 = int(round(float(row.t0)))
        for rel_bin in range(spec["left"], spec["right"]):
            rows.append({
                "session_id": row.session_id,
                "trial_id": float(row.trial_id),
                "t0_event_name": row.t0_event_name,
                "event_label": spec["label"],
                "rel_bin": int(rel_bin),
                "rel_time_s": float(rel_bin * BIN_SECONDS),
                "from_ephys_timestamp": float(t0 + rel_bin * BIN_US),
                "exposure": BIN_SECONDS,
            })
    return pd.DataFrame.from_records(rows)


fr40_counts = fr40[["session_id", "from_ephys_timestamp", "to_ephys_timestamp", *unit_cols]].copy()
fr40_counts["from_ephys_timestamp"] = pd.to_numeric(fr40_counts["from_ephys_timestamp"], errors="coerce")
for unit in unit_cols:
    fr40_counts[unit] = _safe_counts(fr40_counts[unit] / 25.0)

event_index = _expand_event_index(t0_events)
event_predictors = behavior_40_features.drop(columns=[c for c in _unit_cols(behavior_40_features)], errors="ignore")
event_bin_table = (
    event_index
    .merge(fr40_counts.drop(columns="to_ephys_timestamp"), on=["session_id", "from_ephys_timestamp"], how="inner")
    .merge(event_predictors.drop(columns=["trial_id", *task_cols, "trial_number", "trial_number_z"], errors="ignore"), on=["session_id", "from_ephys_timestamp"], how="left")
    .merge(trial_task_table, on=["session_id", "trial_id"], how="left")
)

track_fr_core = fr_track[["session_id", "trial_id", "from_position_bin", "bin_length", *unit_cols]].copy()
track_fr_core["trial_id"] = pd.to_numeric(track_fr_core["trial_id"], errors="coerce")
track_fr_core["from_position_bin"] = pd.to_numeric(track_fr_core["from_position_bin"], errors="coerce")
track_fr_core["exposure"] = pd.to_numeric(track_fr_core["bin_length"], errors="coerce")

track_predictor_cols = [c for c in behavior_track_features.columns if c not in {"cue", "choice_R1", "choice_R2", "trial_outcome", *task_cols, "trial_number", "trial_number_z"}]
track_bin_table = (
    track_fr_core
    .merge(behavior_track_features[track_predictor_cols], on=["session_id", "trial_id", "from_position_bin"], how="left")
    .merge(trial_task_table, on=["session_id", "trial_id"], how="left", suffixes=("", "_trial"))
)
for unit in unit_cols:
    track_bin_table[unit] = _safe_counts(track_bin_table[unit] * track_bin_table["exposure"])

base_predictors = ["trial_number_z"]
track_feature_groups = _available_feature_groups(track_bin_table, FEATURE_GROUPS)
event_feature_groups = _available_feature_groups(event_bin_table, FEATURE_GROUPS)
track_behavior_predictors = sorted({c for cols in track_feature_groups.values() for c in cols})

valid_event = event_bin_table["exposure"].gt(0) & event_bin_table["trial_id"].notna()
valid_track = track_bin_table["exposure"].gt(0) & track_bin_table["trial_id"].notna()
event_bin_table = event_bin_table.loc[valid_event].reset_index(drop=True)
track_bin_table = track_bin_table.loc[valid_track].reset_index(drop=True)

position_source = track_bin_table.copy()
position_source["position_10cm"] = (np.floor(position_source["from_position_bin"] / POSITION_BIN_SIZE_CM) * POSITION_BIN_SIZE_CM).astype("Int64")
position_predictors = sorted(set(track_behavior_predictors + base_predictors + ["trial_number"]))
position_agg = {unit: "sum" for unit in unit_cols}
position_agg.update({"exposure": "sum", "from_position_bin": "mean"})
position_agg.update({col: "mean" for col in position_predictors if col in position_source.columns})
position_model_table = (
    position_source
    .groupby(["session_id", "trial_id", "position_10cm"], as_index=False)
    .agg(position_agg)
)
position_dummies = pd.get_dummies(position_model_table["position_10cm"], prefix="pos", dtype=float, drop_first=True)
position_model_table = pd.concat([position_model_table, position_dummies], axis=1)
position_feature_groups = {"position": position_dummies.columns.tolist()}

print(f"Event-bin table: {event_bin_table.shape[0]:,} rows x {event_bin_table.shape[1]:,} columns")
print(f"Track-bin table: {track_bin_table.shape[0]:,} rows x {track_bin_table.shape[1]:,} columns")
print(f"Position model table: {position_model_table.shape[0]:,} rows x {position_model_table.shape[1]:,} columns")
print("Event feature groups:", {k: len(v) for k, v in event_feature_groups.items()})
print("Track feature groups:", {k: len(v) for k, v in track_feature_groups.items()})
display(track_bin_table[["session_id", "trial_id", "from_position_bin", "exposure", *task_cols]].head())
display(event_bin_table[["session_id", "trial_id", "event_label", "rel_bin", "exposure", *task_cols]].head())


## Negative-Binomial GLM Utilities

Fit full and reduced NB-GLMs, using trial-wise cross-validation and fold-wise imputation/scaling of predictors.


In [ ]:

FIT_ERRORS = (ValueError, np.linalg.LinAlgError, PerfectSeparationError)


def estimate_alpha(y):
    y = np.asarray(y, dtype=float)
    mu = float(np.mean(y))
    var = float(np.var(y, ddof=1)) if y.size > 1 else 0.0
    return float(max((var - mu) / (mu ** 2), 1e-8)) if mu > 0 and var > mu else 1e-8


def nb_deviance(y, mu, alpha):
    family = sm.families.NegativeBinomial(alpha=float(max(alpha, 1e-8)))
    return float(family.deviance(np.asarray(y, dtype=float), np.clip(np.asarray(mu, dtype=float), 1e-9, None)))


def active_columns(df, columns):
    active = []
    for col in columns:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors="coerce")
        if values.notna().sum() >= 2 and values.nunique(dropna=True) > 1:
            active.append(col)
    return active


def make_design(train_df, ref_df, predictors):
    train_x = pd.DataFrame(index=train_df.index)
    ref_x = pd.DataFrame(index=ref_df.index)
    for col in predictors:
        train_col = pd.to_numeric(train_df[col], errors="coerce")
        ref_col = pd.to_numeric(ref_df[col], errors="coerce")
        fill = float(train_col.mean())
        train_col = train_col.fillna(fill)
        ref_col = ref_col.fillna(fill)
        if col in CONTINUOUS_COLS:
            scale = float(train_col.std(ddof=0))
            scale = scale if np.isfinite(scale) and scale > 0 else 1.0
            train_col = (train_col - fill) / scale
            ref_col = (ref_col - fill) / scale
        train_x[col] = train_col
        ref_x[col] = ref_col
    if train_x.shape[1] == 0:
        train_x = pd.DataFrame({"const": np.ones(len(train_df))}, index=train_df.index)
        ref_x = pd.DataFrame({"const": np.ones(len(ref_df))}, index=ref_df.index)
    else:
        train_x = sm.add_constant(train_x, has_constant="add")
        ref_x = sm.add_constant(ref_x, has_constant="add")
    return train_x, ref_x


def fit_nb(train_df, predictors, ref_df):
    predictors = active_columns(train_df, predictors)
    train_x, ref_x = make_design(train_df, ref_df, predictors)
    y = train_df["spike_count"].to_numpy(dtype=float)
    alpha = estimate_alpha(y)
    offset = np.log(np.clip(train_df["exposure"].to_numpy(dtype=float), 1e-9, None))
    result = sm.GLM(y, train_x, family=sm.families.NegativeBinomial(alpha=alpha), offset=offset).fit(maxiter=200, disp=0)
    return result, ref_x, alpha, predictors


def trial_cv_splits(df):
    trials = pd.Series(df["trial_id"].dropna().unique())
    if len(trials) < MIN_TRIALS:
        return []
    trials = trials.sample(frac=1, random_state=RANDOM_STATE).to_numpy()
    n_splits = min(N_CV_FOLDS, len(trials))
    folds = np.array_split(trials, n_splits)
    splits = []
    for test_trials in folds:
        test_mask = df["trial_id"].isin(test_trials).to_numpy()
        train_mask = ~test_mask
        if train_mask.sum() >= MIN_ROWS and test_mask.sum() > 0:
            splits.append((train_mask, test_mask))
    return splits


def lr_pvalue(full_res, reduced_res):
    lr_stat = max(float(2.0 * (full_res.llf - reduced_res.llf)), 0.0)
    df_diff = float(full_res.df_model - reduced_res.df_model)
    return float(stats.chi2.sf(lr_stat, df_diff)) if df_diff > 0 else np.nan


def fit_feature_groups(df, feature_groups, baseline_cols):
    df = df.dropna(subset=["trial_id", "spike_count", "exposure"]).copy()
    df = df[df["exposure"].gt(0) & df["spike_count"].ge(0)]
    n_trials = int(df["trial_id"].nunique())
    mean_count = float(df["spike_count"].mean())
    if len(df) < MIN_ROWS or n_trials < MIN_TRIALS or mean_count < MIN_MEAN_COUNT or df["spike_count"].var() <= 0:
        return []

    baseline = active_columns(df, baseline_cols)
    groups = {name: active_columns(df, cols) for name, cols in feature_groups.items()}
    groups = {name: cols for name, cols in groups.items() if cols}
    full_cols = baseline + sorted({c for cols in groups.values() for c in cols})
    if not groups:
        return []

    splits = trial_cv_splits(df)
    if not splits:
        return []

    dev_full = 0.0
    dev_null = 0.0
    dev_reduced = {name: 0.0 for name in groups}
    valid_reduced = {name: 0 for name in groups}
    fold_alphas = []

    for train_mask, test_mask in splits:
        train_df = df.loc[train_mask]
        test_df = df.loc[test_mask]
        try:
            null_res, null_x, null_alpha, _ = fit_nb(train_df, baseline, test_df)
            full_res, full_x, full_alpha, used_full = fit_nb(train_df, full_cols, test_df)
        except FIT_ERRORS:
            return []

        test_offset = np.log(np.clip(test_df["exposure"].to_numpy(dtype=float), 1e-9, None))
        y_test = test_df["spike_count"].to_numpy(dtype=float)
        mu_null = np.clip(null_res.predict(null_x, offset=test_offset), 1e-9, None)
        mu_full = np.clip(full_res.predict(full_x, offset=test_offset), 1e-9, None)
        dev_null += nb_deviance(y_test, mu_null, null_alpha)
        dev_full += nb_deviance(y_test, mu_full, full_alpha)
        fold_alphas.append(full_alpha)

        for name, cols in groups.items():
            held_out_cols = [c for c in cols if c in used_full]
            if not held_out_cols:
                continue
            reduced_cols = [c for c in used_full if c not in held_out_cols]
            try:
                red_res, red_x, red_alpha, _ = fit_nb(train_df, reduced_cols, test_df)
            except FIT_ERRORS:
                continue
            mu_red = np.clip(red_res.predict(red_x, offset=test_offset), 1e-9, None)
            dev_reduced[name] += nb_deviance(y_test, mu_red, red_alpha)
            valid_reduced[name] += 1

    if not np.isfinite(dev_null) or dev_null <= MIN_NULL_DEVIANCE or not np.isfinite(dev_full):
        return []

    full_fde = 1.0 - dev_full / dev_null
    if not np.isfinite(full_fde) or abs(full_fde) > MAX_ABS_FDE:
        return []
    try:
        full_res, _, alpha_full, used_full = fit_nb(df, full_cols, df)
    except FIT_ERRORS:
        return []

    records = []
    for name, cols in groups.items():
        used_group = [c for c in cols if c in used_full]
        if valid_reduced[name] != len(splits) or not used_group or not np.isfinite(dev_reduced[name]):
            continue
        reduced_fde = 1.0 - dev_reduced[name] / dev_null
        delta_fde = full_fde - reduced_fde
        if not np.isfinite(reduced_fde) or not np.isfinite(delta_fde) or abs(reduced_fde) > MAX_ABS_FDE or abs(delta_fde) > MAX_ABS_FDE:
            continue
        reduced_cols = [c for c in used_full if c not in used_group]
        try:
            reduced_res, _, _, _ = fit_nb(df, reduced_cols, df)
            p_value = lr_pvalue(full_res, reduced_res)
        except FIT_ERRORS:
            p_value = np.nan
        records.append({
            "feature_group": name,
            "family": "negbin",
            "n_rows": int(len(df)),
            "n_trials": n_trials,
            "mean_count": float(df["spike_count"].mean()),
            "var_count": float(df["spike_count"].var(ddof=1)),
            "alpha_nb": float(alpha_full),
            "full_fde_cv": float(full_fde),
            "reduced_fde_cv": float(reduced_fde),
            "delta_fde": float(delta_fde),
            "p_value": p_value,
            "n_predictors_full": int(len(used_full)),
            "n_predictors_group": int(len(used_group)),
        })
    return records


### Parallel Notes

- Leave `PARALLEL_START_METHOD = "fork"` on Linux so worker processes share the large model tables copy-on-write.
- Default `N_WORKERS` now uses most of the server (`CPU_COUNT - 16`, capped at 176 on a 192-core machine). Try 144, 160, or 176; keep the fastest stable setting.
- Set `FORCE_REFIT = True` if you changed model settings and want to ignore cached raw panel results.


## Run Track, Event, And Position Screens (Parallel)

Each screen still fits one neuron at a time statistically, but independent unit/bin work is sharded across worker processes. Result caching lets you resume without redoing completed full-panel screens.


In [ ]:
def _unique(items):
    return list(dict.fromkeys([x for x in items if x in track_bin_table.columns or x in event_bin_table.columns or x in position_model_table.columns]))


def unit_model_frame(base_df, unit, predictors, group_keys):
    keep = _unique(["session_id", "trial_id", "exposure", "trial_number", "trial_number_z", *group_keys, *predictors])
    out = base_df[keep].copy()
    out["unit"] = unit
    out["brain_region"] = [region_lookup.get((s, unit), "Unknown") for s in out["session_id"]]
    out["spike_count"] = pd.to_numeric(base_df[unit], errors="coerce")
    return out


GLM_SETTINGS = {
    "continuous_cols": tuple(CONTINUOUS_COLS),
    "min_trials": MIN_TRIALS,
    "min_rows": MIN_ROWS,
    "min_mean_count": MIN_MEAN_COUNT,
    "min_null_deviance": MIN_NULL_DEVIANCE,
    "max_abs_fde": MAX_ABS_FDE,
    "n_cv_folds": N_CV_FOLDS,
    "random_state": RANDOM_STATE,
}


def _cache_path(panel, max_fits):
    fit_tag = "full" if max_fits is None else f"max{int(max_fits)}"
    settings_tag = f"cv{N_CV_FOLDS}_mintr{MIN_TRIALS}_minrow{MIN_ROWS}_mean{MIN_MEAN_COUNT:g}"
    return RESULT_CACHE_DIR / f"{panel}_{fit_tag}_{settings_tag}_raw.pkl"


def run_screen_cached(base_df, unit_columns, group_keys, feature_groups, panel, max_fits=None, baseline_cols=None):
    cache_path = _cache_path(panel, max_fits)
    if USE_RESULT_CACHE and cache_path.exists() and not FORCE_REFIT:
        print(f"[{panel}] loading cached raw results: {cache_path}")
        return pd.read_pickle(cache_path)

    result = run_screen_parallel(
        base_df=base_df,
        unit_columns=unit_columns,
        group_keys=group_keys,
        feature_groups=feature_groups,
        panel=panel,
        settings=GLM_SETTINGS,
        region_lookup=region_lookup,
        max_fits=max_fits,
        baseline_cols=baseline_cols or base_predictors,
        n_workers=N_WORKERS,
        groups_per_task=GROUPS_PER_TASK,
        start_method=PARALLEL_START_METHOD,
        progress_every=PROGRESS_EVERY,
    )
    if USE_RESULT_CACHE:
        RESULT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        result.to_pickle(cache_path)
        print(f"[{panel}] saved raw results: {cache_path}")
    return result


print(f"CPU count: {CPU_COUNT:,}; using {N_WORKERS:,} workers with BLAS/OpenMP threads capped at 1 per worker.")
print(f"Result cache: {RESULT_CACHE_DIR} | use={USE_RESULT_CACHE} force_refit={FORCE_REFIT}")

track_results_raw = run_screen_cached(
    track_bin_table,
    unit_cols,
    ["session_id", "from_position_bin"],
    track_feature_groups,
    panel="track",
    max_fits=MAX_TRACK_FITS,
)
event_results_raw = run_screen_cached(
    event_bin_table,
    unit_cols,
    ["session_id", "t0_event_name", "event_label", "rel_bin", "rel_time_s"],
    event_feature_groups,
    panel="event",
    max_fits=MAX_EVENT_FITS,
)
position_results_raw = run_screen_cached(
    position_model_table,
    unit_cols,
    ["session_id"],
    position_feature_groups,
    panel="position",
    max_fits=MAX_POSITION_FITS,
    baseline_cols=base_predictors + track_behavior_predictors,
)

track_results = add_panel_fdr_parallel(track_results_raw, ["panel"], FDR_ALPHA)
event_results = add_panel_fdr_parallel(event_results_raw, ["panel", "t0_event_name"], FDR_ALPHA)
position_results = add_panel_fdr_parallel(position_results_raw, ["panel"], FDR_ALPHA)
all_feature_results = pd.concat([track_results, event_results, position_results], ignore_index=True)

display(all_feature_results.head(12))


## Model Diagnostics

Check count overdispersion and how many eligible fits were produced in each analysis panel.


In [ ]:

diag_df = all_feature_results.drop_duplicates(["panel", "session_id", "unit", "feature_group", "from_position_bin", "t0_event_name", "rel_bin"], keep="first") if not all_feature_results.empty else all_feature_results.copy()

fig_diag = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=("Mean-variance structure", "NB dispersion", "Eligible result rows"),
    horizontal_spacing=0.12,
)

for panel, panel_df in diag_df.groupby("panel", sort=False):
    fig_diag.add_trace(
        go.Scatter(
            x=panel_df["mean_count"],
            y=panel_df["var_count"],
            mode="markers",
            name=panel,
            marker=dict(size=5, opacity=0.45),
            customdata=np.stack([panel_df["unit"], panel_df["feature_group"], panel_df["full_fde_cv"].round(4)], axis=1),
            hovertemplate="Unit=%{customdata[0]}<br>Feature=%{customdata[1]}<br>mean=%{x:.3f}<br>var=%{y:.3f}<br>full FDE=%{customdata[2]:.4f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

max_mv = float(np.nanmax(diag_df[["mean_count", "var_count"]].to_numpy())) if len(diag_df) else 1.0
fig_diag.add_trace(go.Scatter(x=np.linspace(1e-3, max_mv, 200), y=np.linspace(1e-3, max_mv, 200), mode="lines", line=dict(color="black", dash="dash"), showlegend=False), row=1, col=1)
fig_diag.add_trace(go.Histogram(x=diag_df["alpha_nb"], nbinsx=50, marker_color="#277da1", showlegend=False), row=1, col=2)
count_summary = diag_df.groupby("panel", as_index=False).size().rename(columns={"size": "n_result_rows"})
fig_diag.add_trace(go.Bar(x=count_summary["panel"], y=count_summary["n_result_rows"], marker_color="#595959", showlegend=False), row=1, col=3)
fig_diag.update_xaxes(type="log", title_text="Mean count", row=1, col=1)
fig_diag.update_yaxes(type="log", title_text="Variance", row=1, col=1)
fig_diag.update_xaxes(title_text="alpha", row=1, col=2)
fig_diag.update_yaxes(title_text="Rows", row=1, col=3)
fig_diag.update_layout(title="Negative-binomial GLM diagnostics", width=1250, height=430)
fig_diag.show()

display(count_summary)


## Track-Binwise Encoding Map

Summarize where behavior and task features are encoded along the track.


In [ ]:

track_summary = (
    track_results.groupby(["feature_group", "from_position_bin"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        n_fits=("unit", "size"),
        n_significant=("is_significant", "sum"),
    )
)
track_x = sorted(track_summary["from_position_bin"].dropna().unique())
track_y = [f for f in FEATURE_ORDER if f in track_summary["feature_group"].unique()]
pivot_sig = track_summary.pivot(index="feature_group", columns="from_position_bin", values="frac_significant").reindex(index=track_y, columns=track_x)
pivot_delta = track_summary.pivot(index="feature_group", columns="from_position_bin", values="median_delta_fde").reindex(index=track_y, columns=track_x)
pivot_n = track_summary.pivot(index="feature_group", columns="from_position_bin", values="n_fits").reindex(index=track_y, columns=track_x)
custom = np.dstack([np.round(pivot_delta.to_numpy(dtype=float), 4), pivot_n.to_numpy(dtype=float)])

fig_track = go.Figure(go.Heatmap(
    z=pivot_sig.to_numpy(dtype=float),
    x=track_x,
    y=track_y,
    colorscale="YlOrRd",
    zmin=0,
    zmax=max(0.05, float(np.nanmax(pivot_sig.to_numpy(dtype=float)))) if len(track_summary) else 0.05,
    customdata=custom,
    text=np.vectorize(lambda x: f"{100*x:.0f}%" if np.isfinite(x) else "")(pivot_sig.to_numpy(dtype=float)),
    texttemplate="%{text}",
    hovertemplate="Position=%{x} cm<br>Feature=%{y}<br>Frac significant=%{z:.3f}<br>Median delta FDE=%{customdata[0]:.4f}<br>n fits=%{customdata[1]:.0f}<extra></extra>",
    colorbar=dict(title="Frac sig"),
))
fig_track.update_layout(title="Track-binwise behavior encoding", width=max(1100, 3 * len(track_x)), height=420)
fig_track.update_xaxes(title="Track position bin (cm)")
fig_track.update_yaxes(title="Feature group")
fig_track.show()

display(track_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).head(20))


## Event-Aligned Encoding Map

Summarize feature encoding in 40 ms bins around each event alignment.


In [ ]:

event_summary = (
    event_results.groupby(["t0_event_name", "event_label", "rel_bin", "rel_time_s", "feature_group"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        n_fits=("unit", "size"),
        n_significant=("is_significant", "sum"),
    )
)
event_names = [e for e in EVENT_ORDER if e in event_summary["t0_event_name"].unique()]
fig_event = make_subplots(rows=max(len(event_names), 1), cols=1, subplot_titles=[EVENT_LABELS[e] for e in event_names] or ["No event fits"], vertical_spacing=0.035)

for row_i, event_name in enumerate(event_names, start=1):
    sub = event_summary[event_summary["t0_event_name"] == event_name]
    x_vals = sorted(sub["rel_time_s"].dropna().unique())
    y_vals = [f for f in FEATURE_ORDER if f in sub["feature_group"].unique()]
    piv_sig = sub.pivot(index="feature_group", columns="rel_time_s", values="frac_significant").reindex(index=y_vals, columns=x_vals)
    piv_delta = sub.pivot(index="feature_group", columns="rel_time_s", values="median_delta_fde").reindex(index=y_vals, columns=x_vals)
    piv_n = sub.pivot(index="feature_group", columns="rel_time_s", values="n_fits").reindex(index=y_vals, columns=x_vals)
    fig_event.add_trace(
        go.Heatmap(
            z=piv_sig.to_numpy(dtype=float),
            x=x_vals,
            y=y_vals,
            colorscale="YlGnBu",
            zmin=0,
            zmax=max(0.05, float(np.nanmax(event_summary["frac_significant"]))) if len(event_summary) else 0.05,
            customdata=np.dstack([np.round(piv_delta.to_numpy(dtype=float), 4), piv_n.to_numpy(dtype=float)]),
            hovertemplate="Time=%{x:.2f}s<br>Feature=%{y}<br>Frac significant=%{z:.3f}<br>Median delta FDE=%{customdata[0]:.4f}<br>n fits=%{customdata[1]:.0f}<extra></extra>",
            colorbar=dict(title="Frac sig") if row_i == 1 else None,
            showscale=row_i == 1,
        ),
        row=row_i,
        col=1,
    )
    fig_event.update_xaxes(title_text="Time from event (s)" if row_i == len(event_names) else "", row=row_i, col=1)
    fig_event.update_yaxes(title_text="Feature", row=row_i, col=1)

fig_event.update_layout(title="Event-aligned behavior encoding", width=1120, height=max(360, 230 * max(len(event_names), 1)))
fig_event.show()

display(event_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).head(24))


## Position Encoding

Test whether neurons encode track position after controlling for behavior predictors.


In [ ]:

position_summary = (
    position_results.groupby(["session_id", "brain_region"], as_index=False)
    .agg(
        frac_position_sig=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        n_units=("unit", "nunique"),
        n_sig=("is_significant", "sum"),
    )
)
position_ranked = position_results.sort_values(["is_significant", "delta_fde", "full_fde_cv"], ascending=[False, False, False])

fig_pos = make_subplots(rows=1, cols=2, subplot_titles=("Position encoding by session", "Position encoding by region"), horizontal_spacing=0.15)
session_pos = position_results.groupby("session_id", as_index=False).agg(frac_sig=("is_significant", "mean"), median_delta=("delta_fde", "median"), n_units=("unit", "nunique"))
region_pos = position_results.groupby("brain_region", as_index=False).agg(frac_sig=("is_significant", "mean"), median_delta=("delta_fde", "median"), n_units=("unit", "nunique")).sort_values("frac_sig", ascending=False)
fig_pos.add_trace(go.Bar(x=session_pos["session_id"], y=session_pos["frac_sig"], marker_color="#386641", customdata=np.stack([session_pos["median_delta"], session_pos["n_units"]], axis=1), hovertemplate="Session=%{x}<br>Frac sig=%{y:.3f}<br>Median delta FDE=%{customdata[0]:.4f}<br>n units=%{customdata[1]:.0f}<extra></extra>", showlegend=False), row=1, col=1)
fig_pos.add_trace(go.Bar(x=region_pos["brain_region"], y=region_pos["frac_sig"], marker_color="#6a4c93", customdata=np.stack([region_pos["median_delta"], region_pos["n_units"]], axis=1), hovertemplate="Region=%{x}<br>Frac sig=%{y:.3f}<br>Median delta FDE=%{customdata[0]:.4f}<br>n units=%{customdata[1]:.0f}<extra></extra>", showlegend=False), row=1, col=2)
fig_pos.update_yaxes(title_text="Fraction significant", tickformat=".0%")
fig_pos.update_xaxes(tickangle=-45, row=1, col=1)
fig_pos.update_xaxes(tickangle=-45, row=1, col=2)
fig_pos.update_layout(title="Whole-track position encoding", width=1250, height=470)
fig_pos.show()

display(position_summary.sort_values(["frac_position_sig", "median_delta_fde"], ascending=[False, False]).head(20))
display(position_ranked.head(20))


## Single-Neuron Exemplars

Inspect representative significant neurons with an encoding map and observed-versus-predicted counts for the selected bin.


In [ ]:

def _source_group(row):
    if row.panel == "track":
        base = track_bin_table[(track_bin_table["session_id"] == row.session_id) & (track_bin_table["from_position_bin"] == row.from_position_bin)]
        groups = track_feature_groups
        keys = ["session_id", "from_position_bin"]
        baseline = base_predictors
    elif row.panel == "event":
        base = event_bin_table[
            (event_bin_table["session_id"] == row.session_id)
            & (event_bin_table["t0_event_name"] == row.t0_event_name)
            & (event_bin_table["rel_bin"] == row.rel_bin)
        ]
        groups = event_feature_groups
        keys = ["session_id", "t0_event_name", "event_label", "rel_bin", "rel_time_s"]
        baseline = base_predictors
    else:
        base = position_model_table[position_model_table["session_id"] == row.session_id]
        groups = position_feature_groups
        keys = ["session_id"]
        baseline = base_predictors + track_behavior_predictors
    predictors = sorted({c for cols in groups.values() for c in cols} | set(baseline))
    return unit_model_frame(base, row.unit, predictors, keys), groups, baseline


def _prediction_df(row):
    model_df, groups, baseline = _source_group(row)
    active_groups = {name: active_columns(model_df, cols) for name, cols in groups.items()}
    full_cols = active_columns(model_df, baseline) + sorted({c for cols in active_groups.values() for c in cols})
    target_cols = active_groups.get(row.feature_group, [])
    reduced_cols = [c for c in full_cols if c not in target_cols]
    full_res, full_x, _, _ = fit_nb(model_df, full_cols, model_df)
    red_res, red_x, _, _ = fit_nb(model_df, reduced_cols, model_df)
    offset = np.log(np.clip(model_df["exposure"].to_numpy(dtype=float), 1e-9, None))
    out = model_df.copy().sort_values(["trial_number", "trial_id"])
    out["pred_full"] = np.clip(full_res.predict(full_x, offset=offset), 1e-9, None)
    out["pred_reduced"] = np.clip(red_res.predict(red_x, offset=offset), 1e-9, None)
    return out


exemplar_rows = []
for panel in ["track", "event", "position"]:
    sub = all_feature_results[(all_feature_results["panel"] == panel) & (all_feature_results["is_significant"])].copy()
    if not sub.empty:
        exemplar_rows.append(sub.sort_values(["delta_fde", "full_fde_cv"], ascending=[False, False]).iloc[0])
exemplar_df = pd.DataFrame(exemplar_rows)

fig_ex = make_subplots(
    rows=max(len(exemplar_df), 1),
    cols=2,
    subplot_titles=[f"{r.panel}: {r.feature_group} | {r.unit}" for r in exemplar_df.itertuples(index=False) for _ in range(2)] or ["No significant exemplars", ""],
    horizontal_spacing=0.12,
    vertical_spacing=0.12,
)

if exemplar_df.empty:
    fig_ex.add_trace(go.Scatter(x=[0], y=[0], mode="text", text=["No significant exemplars found"]), row=1, col=1)
else:
    for row_i, row in enumerate(exemplar_df.itertuples(index=False), start=1):
        pred_df = _prediction_df(row)
        same_unit = all_feature_results[(all_feature_results["panel"] == row.panel) & (all_feature_results["session_id"] == row.session_id) & (all_feature_results["unit"] == row.unit)]
        if row.panel == "track":
            map_df = same_unit.pivot_table(index="feature_group", columns="from_position_bin", values="delta_fde", aggfunc="max")
            fig_ex.add_trace(go.Heatmap(z=map_df.to_numpy(dtype=float), x=map_df.columns, y=map_df.index, colorscale="Viridis", colorbar=dict(title="delta FDE") if row_i == 1 else None, showscale=row_i == 1), row=row_i, col=1)
            x_title = "Track position (cm)"
        elif row.panel == "event":
            unit_event = same_unit[same_unit["t0_event_name"] == row.t0_event_name]
            map_df = unit_event.pivot_table(index="feature_group", columns="rel_time_s", values="delta_fde", aggfunc="max")
            fig_ex.add_trace(go.Heatmap(z=map_df.to_numpy(dtype=float), x=map_df.columns, y=map_df.index, colorscale="Viridis", colorbar=dict(title="delta FDE") if row_i == 1 else None, showscale=row_i == 1), row=row_i, col=1)
            x_title = "Time from event (s)"
        else:
            pos_curve = pred_df.groupby("position_10cm", as_index=False).agg(observed=("spike_count", "mean"), predicted=("pred_full", "mean"))
            fig_ex.add_trace(go.Scatter(x=pos_curve["position_10cm"], y=pos_curve["observed"], mode="markers", name="observed", marker_color="#595959", showlegend=row_i == 1), row=row_i, col=1)
            fig_ex.add_trace(go.Scatter(x=pos_curve["position_10cm"], y=pos_curve["predicted"], mode="lines", name="predicted", line=dict(color="#bc4749", width=3), showlegend=row_i == 1), row=row_i, col=1)
            x_title = "Coarse track position (cm)"

        fig_ex.add_trace(go.Scatter(x=pred_df["trial_number"], y=pred_df["spike_count"], mode="markers", name="observed count", marker=dict(color="rgba(70,70,70,0.45)", size=5), showlegend=row_i == 1), row=row_i, col=2)
        fig_ex.add_trace(go.Scatter(x=pred_df["trial_number"], y=pred_df["pred_full"], mode="lines", name="full model", line=dict(color="#386641", width=2), showlegend=row_i == 1), row=row_i, col=2)
        fig_ex.add_trace(go.Scatter(x=pred_df["trial_number"], y=pred_df["pred_reduced"], mode="lines", name="reduced model", line=dict(color="#bc4749", dash="dash", width=2), showlegend=row_i == 1), row=row_i, col=2)
        fig_ex.update_xaxes(title_text=x_title, row=row_i, col=1)
        fig_ex.update_xaxes(title_text="Trial number", row=row_i, col=2)
        fig_ex.update_yaxes(title_text="Feature group", row=row_i, col=1)
        fig_ex.update_yaxes(title_text="Spike count", row=row_i, col=2)

fig_ex.update_layout(title="Representative significant single-neuron GLMs", width=1250, height=max(430, 340 * max(len(exemplar_df), 1)))
fig_ex.show()

display(exemplar_df)


## Concise Findings

Generate a short text summary from the fitted result tables.


In [ ]:

summary_lines = []

if not track_summary.empty:
    top_track = track_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Track bins**: `{top_track.feature_group}` peaked near {top_track.from_position_bin:g} cm "
        f"({100 * top_track.frac_significant:.1f}% significant, median delta FDE {top_track.median_delta_fde:.4f})."
    )

if not event_summary.empty:
    top_event = event_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Event bins**: `{top_event.feature_group}` was strongest around `{top_event.event_label}` at {top_event.rel_time_s:.2f} s "
        f"({100 * top_event.frac_significant:.1f}% significant, median delta FDE {top_event.median_delta_fde:.4f})."
    )

if not position_results.empty:
    frac_position = float(position_results["is_significant"].mean())
    median_position = float(position_results["delta_fde"].median())
    summary_lines.append(
        f"- **Position**: {100 * frac_position:.1f}% of session-unit fits significantly encoded coarse track position "
        f"after behavior controls (median delta FDE {median_position:.4f})."
    )

if not all_feature_results.empty:
    n_sig = int(all_feature_results["is_significant"].sum())
    n_all = int(len(all_feature_results))
    summary_lines.append(f"- **Statistics**: {n_sig:,}/{n_all:,} feature tests passed panel-wise BH-FDR at q < {FDR_ALPHA} with positive held-out delta FDE.")

if not summary_lines:
    summary_lines.append("- No eligible or significant GLM results were produced with the current settings.")

display(Markdown("\n".join(summary_lines)))


## Validation Checks

These assertions catch common table and model failures before interpreting the plots.


In [ ]:

assert set(all_feature_results["family"].dropna().unique()).issubset({"negbin"})
assert track_bin_table["exposure"].gt(0).all()
assert event_bin_table["exposure"].gt(0).all()
assert all_feature_results["q_value"].dropna().between(0, 1).all()
assert all_feature_results["delta_fde"].dropna().map(np.isfinite).all()
assert all_feature_results["delta_fde"].dropna().abs().le(MAX_ABS_FDE).all()
for table_name, table in [("track_bin_table", track_bin_table), ("event_bin_table", event_bin_table)]:
    min_count = min(float(table[u].min()) for u in unit_cols)
    assert min_count >= 0, f"{table_name} contains negative counts"
print("Validation passed.")
